# PRC1.3: Transfer Learning via Fine-tuning for Image Classification (TASK 2)

**Goals:**
This notebook is part of the first programming assignment (PRC1) for the course "Deep Learning for Visual Signal Processing I." In this notebook, you are required to undertake a series of tasks listed in the "2.TASKS" section. 

**Learning Objectives:**
* Understand the concept of transfer learning: Learn how pre-trained deep learning models can be adapted to new classification tasks with minimal training effort.
* Implement model fine-tuning techniques: Modify and train a pre-trained model by freezing and unfreezing layers, replacing classifier heads, and optimizing hyperparameters.
* Evaluate and compare model performance: Analyze the impact of different fine-tuning strategies on model accuracy and efficiency, and explore best practices for transfer learning in deep learning applications.


**Expected Outcomes:**
* Notebooks: Generate separate notebooks for each experiment conducted during this task.
* Report: Submit a concise report (no more than two pages) that adheres to the specified course format, summarizing your findings and analyses.

**Estimated Completion Time:** The tasks are designed to be completed within an estimated timeframe of 2-3 hours when using GPU acceleration.

---

Author1: Sabbatini, Andrea (andrea.sabbatini@estudiante.uam.es)

Author2: Hamdy, Adham (adham.hamdy@estudiante.uam.es)

Author3: Ciurescu, Irina Alexandra (irinaa.ciurescu@estudiante.uam.es)

---
###### Course: Deep Learning for Visual Signal Processing I
###### Master in [Artificial Intelligence for Image Processing and Computer Vision (IPCVai)](https://ipcv.eu/)
######  [Escuela Politécnica Superior](https://www.uam.es/EPS/Home.htm), [Universidad Autónoma de Madrid](https://www.uam.es/)


# 1. Codebase

We use the provided codebase in the notebooks to develop this assignment. It contains essential scripts to access the dataset and establish the training partitions required for the tasks.

### 1-1.Install the required libraries

Check versions of installed software for this tutorial

* Python 3.10 or above
* Pytorch 2.5.1+cu121 or above
* Torchvision 0.20.1+cu121 or above

In [3]:
import torch
import numpy as np

# Reproducibility (note: full determinism on GPU may require extra flags)
torch.manual_seed(0)
np.random.seed(0)

!python --version

import torch
print(torch.__version__)

import torchvision
print(torchvision.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Python 3.12.12
2.9.0+cu126
0.24.0+cu126
Using device: cuda


### 1-2.Utility Functions (do not modify)

This cell installs and imports a set of shared utility functions used throughout the practical assignments.  
These utilities provide common functionality for:

- Model performance evaluation (overall accuracy, per-class accuracy, reports)
- Dataset inspection and class distribution analysis
- Dataset filtering and class selection
- Visualization of class imbalance

The package is maintained in a centralized GitHub repository to ensure consistency across all notebooks.  
Please **do not edit this cell**, as later sections of the tutorial rely on these functions.

In [4]:
!pip install -q git+https://github.com/jcsma/dlvsp-utils.git

from dlvsp_utils.metrics import (
    calculate_accuracy,
    compute_accuracy_stats,
    compute_accuracy_per_class,
    print_accuracy_report,
)

from dlvsp_utils.data import (
    count_samples_dataset,
    select_classes_dataset,
    inspect_dataset_classes,
)
from dlvsp_utils.viz import (
    plot_class_distribution
)

print("✅ DLVSP utils imported: performance analysis and dataset handling ready!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ DLVSP utils imported: performance analysis and dataset handling ready!


# 2. Loading the dataset for the Classification Task
In this section we load the CIFAR-10 dataset with some basic preprocessing and chose a subset of class.

### 2-1.Basic Preprocessing
We define a basic preprocessing obtained by the tutorial.

In [5]:
from torchvision import datasets, transforms

# Define a basic transformation when loading the dataset
transform = transforms.Compose([    
    #transforms.Resize(224), #for selective fine-tuning, we will get better performance if we match the training image size of popular backbones for ImageNet (224x224)    
    transforms.ToTensor(),  # Convert images to tensor format
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]#we get better performance if we match the training image normalization of popular backbones for ImageNet
    )
])

### 2-2.Load CIFAR-10
In this cell, we load the CIFAR-10 dataset (training and test splits) and apply the preprocessing.

In [6]:
# Load CIFAR-10 data
train_full = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_full  = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
class_names = train_full.classes
print(class_names)

100%|██████████| 170M/170M [00:04<00:00, 39.2MB/s] 


['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


### 2-3.Subset of classes
We choose a subset of 4 classes and load the new datasets.

In [7]:
# Define the classes to be used (e.g., 4 classes from CIFAR-10)
selected_class_names = ['dog', 'cat', 'airplane', 'bird']   # e.g., ['cat','dog'] or 10 classes

# Select subset of classes from CIFAR10
train_dataset_cifar, class_names = select_classes_dataset(train_full, selected_class_names)
test_dataset_cifar, _ = select_classes_dataset(test_full, selected_class_names)

num_classes = len(class_names)
print("Selected classes:", class_names)
print("num_classes:", num_classes)
inspect_dataset_classes(train_dataset_cifar, class_names=class_names, header="\nTRAIN:")
inspect_dataset_classes(test_dataset_cifar, class_names=class_names, header="\nTEST:")
print("✅ Datasets are ready!")

Selected classes: ['dog', 'cat', 'airplane', 'bird']
num_classes: 4

TRAIN:
Classes present:
  Class 0 (dog): 5000 samples
  Class 1 (cat): 5000 samples
  Class 2 (airplane): 5000 samples
  Class 3 (bird): 5000 samples

TEST:
Classes present:
  Class 0 (dog): 1000 samples
  Class 1 (cat): 1000 samples
  Class 2 (airplane): 1000 samples
  Class 3 (bird): 1000 samples
✅ Datasets are ready!


# 3. Lightweight backbone
In this section we train a `ResNet18` initialized with ImageNet weights (full fine-tuning). 

### 3-1.Define the ResNet Architecture
We define a `ResNet-18` model using the standard architecture and setting the parameters for full fine-tuning.
The final classification layer is replaced so that the network outputs predictions for the CIFAR-10 selected classes.

In [8]:
import torch.nn as nn
from torchvision import models

# ResNet18 architecture
model_light = models.resnet18(weights='IMAGENET1K_V1')

# Create a new classification head for CIFAR-10
num_ftrs = model_light.fc.in_features
    
# Create a new classification head
# Here the size of each output sample is set to the number of selected classes
model_light.fc = nn.Linear(num_ftrs, num_classes)
for param in model_light.parameters():
    param.requires_grad = True

print("✅ResNet18 architecture defined!")

num_params = sum(p.numel() for p in model_light.parameters())
print("Number of parameters: "+str(num_params))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 166MB/s]


✅ResNet18 architecture defined!
Number of parameters: 11178564


### 3-2.Training setup
In this cell, we define the main training components: hyperparameters, data loaders, the model, the loss function, and the optimizer.

In [9]:
# Training setup for ResNet18 architecture
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

# Hyperparameters
num_workers=4
num_epochs = 10
batch_size=64
lr=0.001
weight_decay=1e-5

# Setup dataloaders
train_loader = DataLoader(train_dataset_cifar, batch_size=batch_size, shuffle=True, num_workers=num_workers)
test_loader = DataLoader(test_dataset_cifar, batch_size=batch_size, shuffle=False, num_workers=num_workers)

# Setup model, loss, and optimizer
model_light = model_light.to(device)  # instantiate custom CNN
criterion_light = nn.CrossEntropyLoss()
optimizer_light = optim.Adam(model_light.parameters(), lr=lr, weight_decay=weight_decay)

print("✅ Training setup ready for ResNet18!")

✅ Training setup ready for ResNet18!


### 3-3.Train the model
Run this cell to train **ResNet-18** (with `weights='IMAGENET1K_V1'`, i.e., **ImageNet pretraining**) for num_epochs epochs using the training loader. After each epoch, we switch to evaluation mode to report train and test accuracy, which helps monitor learning progress and generalization.

At the end, we print an **accuracy report per class** (train and test). This is especially useful to spot classes that the model struggles with (often visible as lower per-class accuracy, even when overall accuracy looks decent).

Moreover, we print the total training time.

In [10]:
import time

# Training loop for the ResNet18
print(f"Running training for ResNet18 network (pretrained, full fine-tuning) in {device} mode for {num_epochs} epochs")

# Start the timer
start_time = time.time()

for epoch in range(num_epochs):
    model_light.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_light.zero_grad()
        outputs = model_light(images)
        loss = criterion_light(outputs, labels)
        loss.backward()
        optimizer_light.step()
        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    # evaluate epoch results
    model_light.eval()
    train_accuracy, train_labels, train_preds = calculate_accuracy(train_loader, model_light)
    test_accuracy, test_labels, test_preds = calculate_accuracy(test_loader, model_light)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, Test Accuracy: {test_accuracy:.2f}%')

# Stop the timer and print training time
end_time = time.time()
duration = end_time - start_time
print("\nTotal training time: "+str(duration)+" seconds")

# Per-class accuracy
print_accuracy_report(train_labels, train_preds,
                     class_names=class_names,
                     header="TRAIN - Accuracy per class (#train samples /total):",
                     samples_per_class=count_samples_dataset(train_dataset_cifar, num_classes=num_classes))

print_accuracy_report(test_labels, test_preds,
                     class_names=class_names,
                     header="\nTEST - Accuracy per class (#test samples /total):",
                     samples_per_class=count_samples_dataset(test_dataset_cifar, num_classes=num_classes))
print("✅ Training completed!")

Running training for ResNet18 network (pretrained, full fine-tuning) in cuda mode for 10 epochs
Epoch [1/10], Loss: 0.7136, Train Accuracy: 79.72%, Test Accuracy: 76.22%
Epoch [2/10], Loss: 0.4902, Train Accuracy: 84.39%, Test Accuracy: 77.67%
Epoch [3/10], Loss: 0.4092, Train Accuracy: 88.63%, Test Accuracy: 80.20%
Epoch [4/10], Loss: 0.3397, Train Accuracy: 90.55%, Test Accuracy: 79.12%
Epoch [5/10], Loss: 0.2685, Train Accuracy: 93.61%, Test Accuracy: 81.22%
Epoch [6/10], Loss: 0.2135, Train Accuracy: 94.94%, Test Accuracy: 80.15%
Epoch [7/10], Loss: 0.1692, Train Accuracy: 96.59%, Test Accuracy: 80.12%
Epoch [8/10], Loss: 0.1565, Train Accuracy: 97.14%, Test Accuracy: 81.20%
Epoch [9/10], Loss: 0.1203, Train Accuracy: 96.60%, Test Accuracy: 78.97%
Epoch [10/10], Loss: 0.1031, Train Accuracy: 98.29%, Test Accuracy: 80.72%

Total training time: 103.40180087089539 seconds
TRAIN - Accuracy per class (#train samples /total):
Overall accuracy (micro, all samples): 98.29%
Mean per-class a

# 4. Heavyweight backbone
In this section we train a `ResNet101` initialized with ImageNet weights (full fine-tuning). 

### 4-1.Define the ResNet101 Architecture
We define a `ResNet-101` model using the standard architecture and setting the parameters for full fine-tuning.
The final classification layer is replaced so that the network outputs predictions for the CIFAR-10 selected classes.

In [11]:
import torch.nn as nn
from torchvision import models

# ResNet architecture
model_heavy = models.resnet101(weights='IMAGENET1K_V1')

# Create a new classification head for CIFAR-10
num_ftrs = model_heavy.fc.in_features
    
# Create a new classification head
# Here the size of each output sample is set to the number of selected classes
model_heavy.fc = nn.Linear(num_ftrs, num_classes)
for param in model_heavy.parameters():
    param.requires_grad = True

print("✅ResNet101 architecture defined!")

num_params = sum(p.numel() for p in model_heavy.parameters())
print("Number of parameters: "+str(num_params))

Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /root/.cache/torch/hub/checkpoints/resnet101-63fe2227.pth


100%|██████████| 171M/171M [00:00<00:00, 197MB/s]  


✅ResNet101 architecture defined!
Number of parameters: 42508356


### 4-2.Training setup
In this cell, we define the main training components: hyperparameters (the same as before), data loaders, the model, the loss function, and the optimizer.

In [12]:
# Training setup for ResNet101 architecture
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

# Hyperparameters
num_workers=4
num_epochs = 10
batch_size=64
lr=0.001
weight_decay=1e-5

# Setup dataloaders
train_loader = DataLoader(train_dataset_cifar, batch_size=batch_size, shuffle=True, num_workers=num_workers)
test_loader = DataLoader(test_dataset_cifar, batch_size=batch_size, shuffle=False, num_workers=num_workers)

# Setup model, loss, and optimizer
model_heavy = model_heavy.to(device)  # instantiate custom CNN
criterion_heavy = nn.CrossEntropyLoss()
optimizer_heavy = optim.Adam(model_heavy.parameters(), lr=lr, weight_decay=weight_decay)

print("✅ Training setup ready for ResNet101!")

✅ Training setup ready for ResNet101!


### 4-3.Train the model
Run this cell to train **ResNet-101** (with `weights='IMAGENET1K_V1'`, i.e., **ImageNet pretraining**) for num_epochs epochs using the training loader. After each epoch, we switch to evaluation mode to report train and test accuracy, which helps monitor learning progress and generalization.

At the end, we print an **accuracy report per class** (train and test). This is especially useful to spot classes that the model struggles with (often visible as lower per-class accuracy, even when overall accuracy looks decent).

Moreover, we print the total training time.

In [13]:
import time

# Training loop for the ResNet101
print(f"Running training for ResNet101 network (pretrained, full fine-tuning) in {device} mode for {num_epochs} epochs")

# Start the timer
start_time = time.time()

for epoch in range(num_epochs):
    model_heavy.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_heavy.zero_grad()
        outputs = model_heavy(images)
        loss = criterion_heavy(outputs, labels)
        loss.backward()
        optimizer_heavy.step()
        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    # evaluate epoch results
    model_heavy.eval()
    train_accuracy, train_labels, train_preds = calculate_accuracy(train_loader, model_heavy)
    test_accuracy, test_labels, test_preds = calculate_accuracy(test_loader, model_heavy)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, Test Accuracy: {test_accuracy:.2f}%')

# Stop the timer and print training time
end_time = time.time()
duration = end_time - start_time
print("\nTotal training time: "+str(duration)+" seconds")

# Per-class accuracy
print_accuracy_report(train_labels, train_preds,
                     class_names=class_names,
                     header="TRAIN - Accuracy per class (#train samples /total):",
                     samples_per_class=count_samples_dataset(train_dataset_cifar, num_classes=num_classes))

print_accuracy_report(test_labels, test_preds,
                     class_names=class_names,
                     header="\nTEST - Accuracy per class (#test samples /total):",
                     samples_per_class=count_samples_dataset(test_dataset_cifar, num_classes=num_classes))
print("✅ Training completed!")

Running training for ResNet101 network (pretrained, full fine-tuning) in cuda mode for 10 epochs
Epoch [1/10], Loss: 0.7558, Train Accuracy: 75.14%, Test Accuracy: 72.88%
Epoch [2/10], Loss: 0.6126, Train Accuracy: 81.71%, Test Accuracy: 76.08%
Epoch [3/10], Loss: 0.4831, Train Accuracy: 88.20%, Test Accuracy: 80.42%
Epoch [4/10], Loss: 0.5774, Train Accuracy: 46.23%, Test Accuracy: 44.88%
Epoch [5/10], Loss: 0.9386, Train Accuracy: 69.99%, Test Accuracy: 67.70%
Epoch [6/10], Loss: 0.6872, Train Accuracy: 78.32%, Test Accuracy: 73.75%
Epoch [7/10], Loss: 0.5485, Train Accuracy: 82.11%, Test Accuracy: 75.00%
Epoch [8/10], Loss: 0.4455, Train Accuracy: 85.88%, Test Accuracy: 75.28%
Epoch [9/10], Loss: 0.5749, Train Accuracy: 66.06%, Test Accuracy: 64.97%
Epoch [10/10], Loss: 0.6393, Train Accuracy: 83.29%, Test Accuracy: 76.92%

Total training time: 363.19059777259827 seconds
TRAIN - Accuracy per class (#train samples /total):
Overall accuracy (micro, all samples): 83.29%
Mean per-class 